In [5]:
import os
import json
import pandas as pd

from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [6]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)

giga_key = os.getenv("GIGA_KEY")

if not giga_key:

    raise ValueError("Ключ GIGA_KEY не найден в .env")

print("Ключ найден:", giga_key[:8], "...")

Ключ найден: MDE5ZGMz ...


In [8]:
llm = GigaChat(

    credentials=giga_key,

    scope="GIGACHAT_API_PERS",

    model="GigaChat",

    verify_ssl_certs=False,

    temperature=0.2,

    max_tokens=1000,

    timeout=60

)

response = llm.invoke("Привет! Ответь одним коротким предложением.")

print(response.content)

Привет! Коротко и по делу.


In [10]:
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.

Текст заявки: {text}

Верни только число — целое число, соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.

Количество человек:
"""
)

chain = basic_prompt | llm | StrOutputParser()

In [11]:


df = pd.read_csv("rental_26.csv", sep=";")

test_texts = {

    i + 1: text

    for i, text in enumerate(df["text"].head(15))

}
for number, text in test_texts.items():

    result = chain.invoke({"text": text})

    print(f"Заявка №{number}")

    print(f"Текст: {text}")

    print(f"Результат: {result}")

    print("---")


Заявка №1
Текст: Снимем жильё с 1.09по 8.09 двухместный,су в номере ,олимпийская деревня или рядом ,предложения в лс.
Результат: 2
---
Заявка №2
Текст: Ищем недорогое жилье недалеко от моря. 3-местный и 2-местный эконом. 3 взрослых и 3 детей (2,9,11 лет). Строго с 20 по 30 июля
Результат: 6
---
Заявка №3
Текст: Здравствуйте,ищем жилье. 2х местный номер,с удобствами. с 17.07 по 27.07.
Результат: 2
---
Заявка №4
Текст: Здравствуйте. Интересует жилье 3 местный номер.с20 .06 по 28.06 не далеко от моря.
Результат: 3
---
Заявка №5
Текст: Добрый День!! Семья 4 человека, 2 взрослых, дети 10 и 3 года, ищем жилье, можно 3-х местный с доп.местом. С 1 июля поближе к морю и не дорого
😉
Результат: 4
---
Заявка №6
Текст: Здравствуйте. Интересует жильё в Лазаревском. 3е взрослых.
Со своим сан узлом и желательно с балконом. Не больше 10мин до моря.
По приемлемым ценам.
С 4 августа дней на 7-10
Результат: 3
---
Заявка №7
Текст: Ищем жильё эконом класса, до 1000 на двоих, с 7 по 15 августа, недалеко от м

In [12]:
df.head()

,amount,text
0,2,"Снимем жильё с 1.09по 8.09 двухместный,су в но..."
1,6,Ищем недорогое жилье недалеко от моря. 3-местн...
2,2,"Здравствуйте,ищем жилье. 2х местный номер,с уд..."
3,3,Здравствуйте. Интересует жилье 3 местный номер...
4,4,"Добрый День!! Семья 4 человека, 2 взрослых, де..."


In [13]:
df.columns

Index(['amount', 'text'], dtype='str')

In [16]:

results = []

for _, row in df.iterrows():

    text = row["text"]

    try:

        result = chain.invoke({"text": text})

        results.append(result.strip())

    except Exception as e:

        results.append(f"ERROR: {e}")

df["result"] = results

df["result_num"] = pd.to_numeric(df["result"], errors="coerce")

correct = (df["amount"] == df["result_num"]).sum()

total = len(df)

errors = total - correct

accuracy = correct / total


print(f"Всего заявок: {total}")

print(f"Верных ответов: {correct}")

print(f"Ошибок: {errors}")

print(f"Точность: {accuracy:.1%}")


display(df)

df.to_csv("rental_26_with_results.csv", index=False, encoding="utf-8-sig")

Всего заявок: 15
Верных ответов: 15
Ошибок: 0
Точность: 100.0%


,amount,text,result,result_num
0,2,"Снимем жильё с 1.09по 8.09 двухместный,су в но...",2,2
1,6,Ищем недорогое жилье недалеко от моря. 3-местн...,6,6
2,2,"Здравствуйте,ищем жилье. 2х местный номер,с уд...",2,2
3,3,Здравствуйте. Интересует жилье 3 местный номер...,3,3
4,4,"Добрый День!! Семья 4 человека, 2 взрослых, де...",4,4
5,3,Здравствуйте. Интересует жильё в Лазаревском. ...,3,3
6,2,"Ищем жильё эконом класса, до 1000 на двоих, с ...",2,2
7,3,Добрый день. Ищем жильё в Лазаревском. С 10.08...,3,3
8,3,Здравствуйте! Ищем жильё с 15 по 23 августа 2...,3,3
9,5,Добрый день интересует жилье семья 5 человек д...,5,5
